# 06 — Failure Prediction Demo

## Objective
Use the trained model to predict failure risk for a new machine observation.

### Output
- Failure probability
- Risk level
- Predicted class
- Global model-important factors

### Risk levels used in this academic demo
- **LOW:** probability < 0.30
- **MEDIUM:** 0.30 ≤ probability < 0.70
- **HIGH:** probability ≥ 0.70

These thresholds are project-defined demonstration thresholds. In a real factory, they should be selected using business costs, maintenance capacity, safety requirements, and historical validation.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_DIR = PROJECT_ROOT / "models"
IMPORTANCE_PATH = PROJECT_ROOT / "reports" / "feature_importance_encoded.csv"

preprocessor = joblib.load(MODEL_DIR / "preprocessor.pkl")
model = joblib.load(MODEL_DIR / "random_forest_model.pkl")

# Original input schema used during training.
FEATURE_COLUMNS = [
    "Temperature_C",
    "Pressure_bar",
    "Vibration_mm_s",
    "Rotational_Speed_rpm",
    "Voltage_V",
    "Current_A",
    "Operating_Hours",
    "Tool_Wear_Min",
    "Machine_Type",
    "Maintenance_History_Count"
]

importance_df = pd.read_csv(IMPORTANCE_PATH)

In [6]:
def risk_level(probability):
    """Convert failure probability into a project-defined risk category."""
    if probability < 0.30:
        return "LOW"
    elif probability < 0.70:
        return "MEDIUM"
    return "HIGH"


def predict_machine(machine_data):
    """
    Predict failure risk for one machine.

    Parameters
    ----------
    machine_data : dict
        Dictionary containing all FEATURE_COLUMNS.

    Returns
    -------
    dict
        Prediction result with probability, class and risk level.
    """
    row = pd.DataFrame([machine_data], columns=FEATURE_COLUMNS)
    transformed = preprocessor.transform(row)

    probability = float(model.predict_proba(transformed)[0, 1])
    prediction = int(probability >= 0.50)

    return {
        "Failure_Probability": probability,
        "Failure_Prediction": prediction,
        "Risk_Level": risk_level(probability)
    }

In [7]:
# Example machine observation
example_machine = {
    "Temperature_C": 49.0,
    "Pressure_bar": 9.8,
    "Vibration_mm_s": 8.2,
    "Rotational_Speed_rpm": 1550,
    "Voltage_V": 226,
    "Current_A": 21.5,
    "Operating_Hours": 11200,
    "Tool_Wear_Min": 225,
    "Machine_Type": "M",
    "Maintenance_History_Count": 1
}

result = predict_machine(example_machine)

print(f"Failure Probability: {result['Failure_Probability']:.2%}")
print(f"Failure Prediction : {result['Failure_Prediction']} (1 = failure risk)")
print(f"Risk Level         : {result['Risk_Level']}")

Failure Probability: 88.58%
Failure Prediction : 1 (1 = failure risk)
Risk Level         : HIGH


In [8]:
# Show the globally important original factors.
# Machine_Type may appear as one-hot encoded features; we collapse those labels conceptually.
def original_factor_name(encoded_name):
    name = encoded_name.replace("num__", "").replace("cat__", "")
    if name.startswith("Machine_Type_"):
        return "Machine_Type"
    return name

importance_df["Original_Factor"] = importance_df["Encoded_Feature"].apply(original_factor_name)

factor_summary = (
    importance_df
    .groupby("Original_Factor", as_index=False)["Importance_Mean"]
    .sum()
    .sort_values("Importance_Mean", ascending=False)
)

print("Top model-important factors:")
display(factor_summary.head(5))

Top model-important factors:


,Original_Factor,Importance_Mean
3,Operating_Hours,0.722636
7,Tool_Wear_Min,0.050658
8,Vibration_mm_s,0.031658
6,Temperature_C,0.008385
5,Rotational_Speed_rpm,0.005294


## Example report output format

```text
Failure Probability: 87.00%
Failure Prediction : 1
Risk Level         : HIGH

Important Factors:
1. Temperature
2. Vibration
3. Tool Wear
4. Operating Hours
5. Pressure
```

**Important:** The actual ranking should come from your notebook output. Do not manually claim the example order unless the trained model produces it.


## How the Prediction Demo Works

1. Load the saved preprocessing pipeline and trained model.
2. Accept the required machine sensor and operational features.
3. Transform the input using the fitted preprocessing pipeline.
4. Predict the failure class and its estimated probability.
5. Map the probability to a project-defined risk category.
6. Display globally important model features.

## Risk Categories

- LOW: probability below 0.30
- MEDIUM: probability from 0.30 to below 0.70
- HIGH: probability of 0.70 or higher

These thresholds are illustrative and are not validated
industrial maintenance or safety thresholds.

## Limitations

The predicted probability is model-dependent and should not be
interpreted as a calibrated real-world failure probability unless
calibration and external validation have been performed.
The globally important factors are not necessarily the causes
of an individual prediction.
